In [ ]:
# Cell 1 — Load config and utilities
%run /home/jovyan/work/setup/config.py
import sys; sys.path.insert(0, "/home/jovyan/work")
from utils.dq import dq_check, write_dq_log
from utils.delta_utils import save_layer

In [ ]:
# Cell 2 — Read raw CSVs via pandas (Spark UTF-16LE reader produces nulls)
import os
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

pdf_sales = (
    pd.read_csv(SALES_FILE, sep="\t", encoding="utf-16", dtype=str, keep_default_na=False)
    .rename(columns={"$ Volume": "dollar_volume_raw"})
    .where(lambda df: df.notna(), other=None)
)
pdf_channel = (
    pd.read_csv(CHANNEL_FILE, sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
    .where(lambda df: df.notna(), other=None)
)

df_sales = (
    spark.createDataFrame(pdf_sales)
    .withColumn("_ingest_ts",   current_timestamp())
    .withColumn("_source_file", lit(os.path.basename(SALES_FILE)))
)
df_channel = (
    spark.createDataFrame(pdf_channel)
    .withColumn("_ingest_ts",   current_timestamp())
    .withColumn("_source_file", lit(os.path.basename(CHANNEL_FILE)))
)

print(f"Sales rows: {df_sales.count()} | Channel rows: {df_channel.count()}")
print("Sales columns:", df_sales.columns)

In [ ]:
# Cell 3 — Append to Delta Bronze + PostgreSQL bronze schema
# Bronze is append-only: each run adds a new timestamped batch.
# Silver always rebuilds from the latest batch (max _ingest_ts).
save_layer(df_sales,   "bronze_beverage_sales", BRONZE_PATH, PG_WRITE_PROPS,
           pg_schema="bronze", delta_mode="append", pg_mode="append")
save_layer(df_channel, "bronze_channel_group",  BRONZE_PATH, PG_WRITE_PROPS,
           pg_schema="bronze", delta_mode="append", pg_mode="append")
print("Bronze layer complete")

In [ ]:
# Cell 4 — DQ Bronze checks (against the current batch, not the cumulative table)
import uuid
from pyspark.sql.functions import col

run_id = str(uuid.uuid4())
checks = [
    dq_check(run_id, "bronze", "bronze_beverage_sales", "row_count_gt_0",
             "True", df_sales.count() > 0),
    dq_check(run_id, "bronze", "bronze_channel_group",  "row_count_gt_0",
             "True", df_channel.count() > 0),
    dq_check(run_id, "bronze", "bronze_beverage_sales", "col_count",
             "15", len(df_sales.columns)),
    dq_check(run_id, "bronze", "bronze_beverage_sales", "no_null_date",
             "0", df_sales.filter(col("DATE").isNull()).count()),
    dq_check(run_id, "bronze", "bronze_beverage_sales", "no_null_brand_flvr",
             "0", df_sales.filter(col("CE_BRAND_FLVR").isNull()).count()),
]
write_dq_log(spark, checks, GOLD_PATH)